# Notebook 02 — Environmental Datasets
**GeoAI for Climate, Environment and Health** | 25 June 2026

---

## Learning objectives
- Understand where climate and environmental data comes from (satellite products, reanalysis)
- Load and explore a gridded environmental dataset
- Read spatial subplots showing multiple variables at once
- Use a nearest-neighbour lookup to attach grid values to point locations

**Time:** ~20 minutes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib

def _find_workshop_dir():
    here = pathlib.Path().resolve()
    for candidate in (here, here.parent, here.parent.parent):
        if (candidate / 'data').exists():
            return candidate
    return here

WORKSHOP_DIR = _find_workshop_dir()  # works whether run flat or from a notebooks/ subfolder
DATA_DIR = WORKSHOP_DIR / 'data'
sys.path.insert(0, str(WORKSHOP_DIR))

from shared_utils import find_nearest_grid_cells
print('Ready.')

---
## 📖 1. Where does environmental data come from?

In settings where ground sensors are limited — which describes most of sub-Saharan Africa — researchers rely on **satellite products** and **reanalysis datasets** to characterise environmental conditions.

| Variable | Typical source | Temporal resolution | Notes |
|---|---|---|---|
| Temperature (mean/max) | **ERA5-Land** (ECMWF reanalysis) | Hourly → aggregated to daily | Combines weather model + observations |
| PM2.5 / air quality | **CAMS** (Copernicus Atmosphere) | 3-hourly → daily | Forecast model + satellite assimilation |
| NDVI (vegetation) | **MODIS** (Terra/Aqua satellites) | 16-day composites | Proxy for vegetation greenness, crop health |
| Precipitation | **CHIRPS** | Daily | Satellite infrared + weather station data |
| Relative humidity | Derived from temperature + dewpoint | Same as temperature source | Magnus-Tetens formula |

All of these products are available free via **Google Earth Engine** — which we explore in Notebook 05.

**What is a grid?**

These datasets divide the Earth's surface into a regular grid of rectangular cells. Each cell has one value per time step. Our workshop dataset uses **0.25° spacing**, which is approximately 27 km × 27 km per cell near Zimbabwe's latitude. That means:
- A city like Harare might fall inside a single grid cell
- Elevation differences of less than 27 km are averaged together

This is called a **gridded environmental product** and it is the foundation of most climate-health research in low-resource settings.

> ⚠️ **NDVI caveat:** Real MODIS NDVI products are 16-day composites, not daily. Our synthetic dataset simplifies this to daily for teaching purposes, but in real analyses you need to account for this temporal resolution.

---
## ✅ Worked example 1 — Load and explore the climate grid

In [ ]:
grid = pd.read_csv(DATA_DIR / 'climate_grid.csv')
print(f'Shape: {grid.shape}')
print(f'Grid cells: {grid["grid_id"].nunique()}')
print(f'Longitude range: {grid["lon"].min():.2f} → {grid["lon"].max():.2f}')
print(f'Latitude range:  {grid["lat"].min():.2f} → {grid["lat"].max():.2f}')
print()
grid.head()

In [ ]:
# Descriptive statistics — what are the typical values?
grid[['temp_mean_c', 'pm25_ugm3', 'ndvi', 'precipitation_mm']].describe().round(2)

In [ ]:
# 2×2 spatial scatter map — four variables at once.
vars_info = [
    ('temp_mean_c',       'Mean temperature (°C)',  'RdYlBu_r'),
    ('pm25_ugm3',         'PM2.5 (µg/m³)',          'YlOrRd'),
    ('ndvi',              'NDVI',                   'YlGn'),
    ('precipitation_mm',  'Precipitation (mm)',     'Blues'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, (col, title, cmap) in zip(axes.ravel(), vars_info):
    sc = ax.scatter(grid['lon'], grid['lat'], c=grid[col], cmap=cmap, s=18, alpha=0.85)
    plt.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

fig.suptitle('Environmental grid — snapshot 2024-07-15 (synthetic)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## 📖 2. How the nearest-cell lookup works

We have **point** locations (facilities, villages, participant residences) and a **grid** of environmental values. To attach the right environmental value to each point, we find the **nearest grid cell centroid** using a *k-d tree* — a data structure that makes nearest-neighbour searches fast even across millions of points.

```
Grid centroids:   × × × × ×
                  × × × × ×
                  × ×   × ×
Your point:             ★
Nearest centroid: ────────→ ×  (closest grid cell)
```

The `find_nearest_grid_cells()` function from `shared_utils.py` does this. It uses `scipy.spatial.cKDTree` under the hood. You pass it a DataFrame of point locations and a grid DataFrame, and it returns your points with a new `grid_id` column attached.

> **Why Euclidean distance on lat/lon is acceptable here:** Over a ~700 × ~650 km study area (Zimbabwe), the error from treating lat/lon as a flat coordinate plane is less than 0.5% — negligible compared to the 27 km grid cell size. For continental-scale studies you would need a proper geodetic distance.

---
## 🧪 Exercise 1 — Mean temperature at district facilities  *(3 points)*

For facilities in **Harare** district only:
1. Use `find_nearest_grid_cells()` to attach a `grid_id` to each facility.
2. Merge onto the grid to get `temp_mean_c` for each facility.
3. Compute the **mean** temperature across those facilities.

Store the result in `harare_mean_temp` (a float).

In [ ]:
facilities = pd.read_csv(DATA_DIR / 'health_facilities.csv')
facilities_mapped = facilities.dropna(subset=['latitude', 'longitude'])

harare = facilities_mapped[facilities_mapped['district'] == 'Harare'].copy()
harare = find_nearest_grid_cells(harare, grid)
harare = harare.merge(grid[['grid_id', 'temp_mean_c']], on='grid_id', how='left')

# Hint: harare now has a temp_mean_c column for each Harare facility.
# Compute the mean of that column and store it in harare_mean_temp.

harare_mean_temp = harare['temp_mean_c'].mean()

print(f'Mean temperature at Harare grid cells: {harare_mean_temp:.2f} °C')


---
### ✅ Knowledge Check — Joining environmental data to facility locations

*No coding here — just check your understanding before moving on.*

**Q1. Why do we attach a facility to its *nearest* environmental grid cell instead of requiring an exact coordinate match?**

- **A.** Environmental data is always recorded at the exact facility location
- **B.** Environmental measurements come on a grid (e.g. satellite pixels), so a facility almost never sits exactly on a grid point — nearest-neighbour matching is how point locations get linked to gridded data
- **C.** Exact matching is faster to compute
- **D.** Nearest-cell matching is only needed when GPS coordinates are wrong

<details>
<summary>Show answer</summary>

**Answer: B** — Gridded environmental products (satellite or model output) report one value per grid cell, not per exact coordinate — nearest-neighbour assignment is the standard way to bridge point data (facilities) to grid data.

</details>

**Q2. If the environmental grid is very coarse (e.g. each cell covers 25 km²), what's a risk of assigning every facility in a district to the few nearest cells?**

- **A.** No risk — coarser grids are always more accurate
- **B.** Facilities that are actually in quite different micro-environments (e.g. near a busy road vs. a forest) may get assigned the same exposure value, hiding real variation
- **C.** Coarse grids make the join run faster, which is the only consequence
- **D.** It would cause the merge to fail

<details>
<summary>Show answer</summary>

**Answer: B** — This is the Modifiable Areal Unit Problem (MAUP) in miniature — coarse spatial resolution can smear out real differences in exposure between nearby locations.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 🧪 Exercise 2 — Highest PM2.5 grid cell  *(2 points)*

Find the grid cell with the **highest** `pm25_ugm3` value. Store its latitude in `max_pm25_lat` and longitude in `max_pm25_lon` (both floats).

In [ ]:
# Hint: grid['pm25_ugm3'].idxmax() gives the row index with the highest
# PM2.5 value. Use .loc[that_index] to read off the 'lat' and 'lon' values
# for max_pm25_lat and max_pm25_lon.

worst_cell = grid.loc[grid['pm25_ugm3'].idxmax()]
max_pm25_lat = float(worst_cell['lat'])
max_pm25_lon = float(worst_cell['lon'])

print(f'Highest PM2.5: {grid["pm25_ugm3"].max():.2f} µg/m³  at lat={max_pm25_lat}, lon={max_pm25_lon}')


---
### ✅ Knowledge Check — Identifying pollution hotspots

*No coding here — just check your understanding before moving on.*

**Q1. What's the practical value of finding the single grid cell with the highest PM2.5 in an exposure dataset?**

- **A.** It tells you the average exposure across the whole region
- **B.** It flags a specific hotspot location that may warrant targeted air-quality interventions or further investigation
- **C.** It has no public health relevance
- **D.** It replaces the need to look at any other grid cells

<details>
<summary>Show answer</summary>

**Answer: B** — Identifying extremes (hotspots) is a standard first step in exposure assessment — it helps prioritise where to look closer or intervene, even though most analysis will use the full distribution, not just the max.

</details>



In [ ]:
# Tests hidden — run your solution above first
pass

---
## 🏁 Recap

- Gridded environmental products (ERA5, CAMS, CHIRPS, MODIS) divide the study area into cells; each cell gets one value per time step.
- A nearest-neighbour lookup with cKDTree attaches the best available environmental estimate to any point location.
- The 2×2 subplot map is a quick way to compare spatial patterns across variables — notice how temperature increases northward, PM2.5 is higher in the west, and NDVI tracks vegetation.

**Next:** Notebook 03 implements the full spatial join from scratch and previews what changes when locations are *time-varying*.